In [1]:
import re
import math
import requests
from bs4 import BeautifulSoup
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
URL = "https://www.gutenberg.org/cache/epub/84/pg84-images.html"

def fetch_gutenberg_text(url: str) -> str:
    html = requests.get(url, timeout=30).text
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text("\n")
    text = text.replace("\u00a0", " ")  # non-breaking spaces

    start_key = "START OF THE PROJECT GUTENBERG EBOOK"
    end_key = "END OF THE PROJECT GUTENBERG EBOOK"
    s = text.find(start_key)
    e = text.find(end_key)
    if s != -1 and e != -1 and e > s:
        text = text[s:e]

    text = re.sub(r"\s+", " ", text).strip()
    return text

raw_text = fetch_gutenberg_text(URL)

In [3]:
TOKEN_RE = r"\b\w+\b|[^\w\s]"

def tokenize(s: str):
    return re.findall(TOKEN_RE, s.lower())

tokens = tokenize(raw_text)
print("num tokens:", len(tokens))


counts = Counter(tokens)
vocab = sorted(counts, key=counts.get, reverse=True)
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)
print("vocab size:", vocab_size)

encoded = [word2idx[t] for t in tokens]


num tokens: 85982
vocab size: 7029


In [4]:
window_size = 100
seq_len = window_size - 1  # 99

X_list, y_list = [], []
for i in range(len(encoded) - window_size):
    X_list.append(encoded[i : i + seq_len])
    y_list.append(encoded[i + seq_len])

X = torch.tensor(X_list, dtype=torch.long)         # (N, 99)
y = torch.tensor(y_list, dtype=torch.long)         # (N,)
print("X shape:", X.shape, "y shape:", y.shape)

X shape: torch.Size([85882, 99]) y shape: torch.Size([85882])


In [5]:
class TextWindowDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return self.X.size(0)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

n = X.size(0)
split = int(0.9 * n)
X_train, y_train = X[:split], y[:split]
X_val, y_val = X[split:], y[split:]

batch_size = 64
train_loader = DataLoader(TextWindowDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TextWindowDataset(X_val, y_val), batch_size=batch_size, shuffle=False)


In [6]:
class LSTMTextGen(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=256, num_layers=1, dropout=0.0):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, state=None):
        # x: (B, 99) long
        x = self.embed(x)                     # (B, 99, E)
        out, state = self.lstm(x, state)      # out: (B, 99, H), state: (h_n, c_n)
        last = out[:, -1, :]                  # (B, H)
        logits = self.fc(last)                # (B, vocab_size)
        return logits, state


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = LSTMTextGen(vocab_size, embed_dim=100, hidden_dim=256, num_layers=1).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    run_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        logits, _ = model(xb)
        loss = loss_fn(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        run_loss += loss.item()

    avg_train = run_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits, _ = model(xb)
            val_loss += loss_fn(logits, yb).item()
    avg_val = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{num_epochs} | train loss: {avg_train:.4f} | val loss: {avg_val:.4f} | ppl: {math.exp(avg_val):.2f}")

# ---------------------------
# 8) Generate text
# ---------------------------
def detokenize(tok_list):
    # simple punctuation spacing cleanup
    punct_attach = {".", ",", "!", "?", ";", ":", ")", "]", "}", "\"", "'", "’"}
    no_space_after = {"(", "[", "{", "\"", "'", "‘"}
    out = ""
    for t in tok_list:
        if not out:
            out = t
        elif t in punct_attach:
            out += t
        elif out[-1] in no_space_after:
            out += t
        else:
            out += " " + t
    return out

@torch.no_grad()
def continue_text(model, seed, steps=180, window=99, temperature=1.0):
    model.eval()
    fallback = word2idx.get("the", 0)

    seed_tokens = tokenize(seed)
    ids = [word2idx.get(t, fallback) for t in seed_tokens]
    ids = ([fallback] * (window - len(ids)) + ids)[-window:]

    out_tokens = seed_tokens[:]
    for _ in range(steps):
        x = torch.tensor([ids], dtype=torch.long, device=device)  # (1,99)
        logits, _ = model(x)                                      # (1,vocab)
        logits = logits[0] / max(temperature, 1e-6)

        probs = torch.softmax(logits, dim=-1)
        next_id = int(torch.multinomial(probs, 1).item())         # sampling (nicer than argmax)
        out_tokens.append(idx2word[next_id])

        ids = ids[1:] + [next_id]

    return detokenize(out_tokens)

seed = "my name is"
gen = continue_text(model, seed, steps=200, temperature=0.9)
print("\n--- GENERATED ---\n")
print(gen)

Epoch 1/10 | train loss: 5.9208 | val loss: 5.6319 | ppl: 279.18
Epoch 2/10 | train loss: 5.0723 | val loss: 5.5007 | ppl: 244.85
Epoch 3/10 | train loss: 4.5218 | val loss: 5.5263 | ppl: 251.22
Epoch 4/10 | train loss: 4.0177 | val loss: 5.6290 | ppl: 278.39
Epoch 5/10 | train loss: 3.5408 | val loss: 5.7363 | ppl: 309.91
Epoch 6/10 | train loss: 3.0941 | val loss: 5.8920 | ppl: 362.13
Epoch 7/10 | train loss: 2.6821 | val loss: 6.0636 | ppl: 429.91
Epoch 8/10 | train loss: 2.3108 | val loss: 6.2510 | ppl: 518.55
Epoch 9/10 | train loss: 1.9778 | val loss: 6.4771 | ppl: 650.10
Epoch 10/10 | train loss: 1.6817 | val loss: 6.6914 | ppl: 805.45

--- GENERATED ---

my name is reverberated - copêt raging uneasy opposition communicated conversation my misery protectors still inflict would sacrifice the greatest danger, or a prophetic promise - echoed, that i have no power of benevolent different from man to bestow and in a very dogs of money, that is less to say, dejection, whose dispositio